# Quantum Oracle Sketching — Matrix Block Encoding

> This is the **second** notebook in the quantum-oracle-sketching series. The [first notebook](qauntum_oracle_sketching.ipynb) sketched Boolean phase oracles and real-valued state-preparation unitaries from streaming classical samples. Here we take the next step: sketching a **block encoding** of an arbitrary (sparse) matrix $A$ — the quantum-input format consumed by [QSVT](https://arxiv.org/abs/1806.01838) [[4]](#ref4), [HHL](https://en.wikipedia.org/wiki/Quantum_algorithm_for_linear_systems_of_equations) [[5]](#ref5), and the LS-SVM and PCA pipelines that follow this notebook.
>
> The construction layers on top of the same per-sample primitive: each classical sample $z_t = (i_t, j_t, A_{i_t j_t})$ contributes a small phase rotation that fires only on the basis state $|i_t, j_t\rangle$. After $M$ samples the cumulative gate is the *matrix-element phase oracle* $V|i,j\rangle = e^{i\theta A_{ij}}|i,j\rangle$. Combined with a Hadamard test and a column-register Hadamard transform, this yields a block encoding $U_A$ with $\bigl(\langle 0^a|\otimes I\bigr) U_A \bigl(|0^a\rangle\otimes I\bigr) = A/\alpha$ for some normalisation $\alpha$.
>
> ---
>
> **Inputs.** $M$ classical samples of the matrix entries $\{(i_t, j_t, A_{i_t j_t})\}_{t=1}^M$ with $(i_t, j_t) \sim \mathrm{Unif}([N]\times[N])$, plus — for sparse matrices — analogous samples for the *matrix-index* oracle (Sec. D.5–D.6 of [[1]](#ref1)).
>
> **Output.** A unitary $U_A$ on $a + 2n$ qubits ($n = \log_2 N$ data qubits, $a$ ancillas) such that $\|(\langle 0^a|\otimes I) U_A (|0^a\rangle\otimes I) - A/\alpha\|_2 \le \epsilon$ in operator norm, with $\alpha = \mathcal{O}(N \cdot \|A\|_\text{max})$ and $\epsilon = \mathcal{O}(N^2/M)$ in diamond distance.
>
> **Sample complexity.** $M_\text{total} = \Theta(N^2 Q^2 / \epsilon)$ classical samples for $Q$ block-encoding queries to total error $\epsilon$ — one factor of $N$ harder than the Boolean / vector case (Sec. D.5 of [[1]](#ref1)), reflecting the $N^2$ matrix entries that must be resolved.
>
> ---
>
> **Keywords:** Quantum Oracle Sketching, Block Encoding, QSVT, Quantum Machine Learning

## Two oracles, one block encoding

The standard quantum-input format for matrix arithmetic — the form consumed by HHL, QSVT, and most quantum-machine-learning routines — is a **block encoding**. A unitary $U_A$ is an $(\alpha, a, \epsilon)$-block encoding of $A \in \mathbb{C}^{N \times N}$ if

$$
\Bigl\| A - \alpha\, \bigl(\langle 0|^{\otimes a}\otimes I_n\bigr)\, U_A\, \bigl(|0\rangle^{\otimes a}\otimes I_n\bigr) \Bigr\|_2 \;\le\; \epsilon,
$$

i.e., $A/\alpha$ sits in the top-left block of $U_A$ when the $a$ ancillas are in $|0\rangle^{\otimes a}$. The Gilyén–Su–Low–Wiebe construction [[4]](#ref4) shows that any $s$-sparse matrix admits such an encoding given two classical-style oracles:

1. **Matrix-element oracle.**

$$
O_A^{(\text{el})} \,|i\rangle\,|j\rangle\,|0\rangle_b \;=\; |i\rangle\,|j\rangle\,|A_{ij}\rangle_b,
$$

or, in *phase* form,

$$
O_A^{(\text{el})} \,|i\rangle\,|j\rangle \;=\; e^{i\,\theta\, A_{ij}}\,|i\rangle\,|j\rangle.
$$

The phase form is the one we sketch here — it is diagonal, so per-sample contributions accumulate coherently exactly as in the Boolean case.

2. **Matrix-index oracle.**

$$
O_A^{(\text{idx})} \,|i\rangle\,|k\rangle \;=\; |i\rangle\,\bigl|j_k(i)\bigr\rangle, \qquad k = 1,\dots,s,
$$

where $j_k(i)$ is the column index of the $k$-th non-zero entry of row $i$. For dense matrices, $s = N$ and the index oracle reduces to the identity. For sparse matrices, this oracle encodes the sparsity pattern.

Combined into the standard sparse-block-encoding circuit (Lemma 48 of [[4]](#ref4)), these two oracles produce $U_A$ with $\alpha = \sqrt{s\, s_c}\,\|A\|_\text{max}$, where $s, s_c$ are the row / column sparsities. The sketched versions retain the same shape — they merely replace each oracle with an empirical approximation built from $M$ classical samples, each processed once and immediately discarded.

For the rest of this notebook we focus on the **matrix-element phase oracle** (the harder of the two) and demonstrate how it composes with a Hadamard test plus a column-register Hadamard transform to yield a complete block encoding. The matrix-index oracle is the analogous sketch for sparse-pattern access; for the dense demonstrations below we treat $s = N$ and skip it.

## Algorithm — sketched matrix-element phase oracle

We aim to implement

$$
O_A^{(\text{el})} \,|i\rangle\,|j\rangle \;=\; e^{i\,\theta\, A_{ij}}\,|i\rangle\,|j\rangle,\qquad i,j \in [N],
$$

given only random classical samples $z_t = (i_t,\,j_t,\,A_{i_t j_t})$ for $t = 1,\dots,M$, where $(i_t, j_t)$ is drawn uniformly from $[N]\times[N]$ (so each of the $N^2$ matrix entries is sampled with probability $1/N^2$).

**Per-sample rotation.** For each sample we apply a 2-register controlled phase that fires only on the basis state $|i_t,j_t\rangle$:

$$
V_t \;=\; \exp\!\Bigl(i\,\tau\, A_{i_t j_t}\,|i_t,j_t\rangle\!\langle i_t,j_t|\,/\,M\Bigr),\qquad \tau = \theta\, N^2.\tag{1}
$$

The factor of $N^2$ in $\tau$ is the matrix analogue of the Boolean-case $\tau = \pi N$: it cancels the empirical-frequency mean $1/N^2$ so that the cumulative phase converges to $\theta\, A_{ij}$.

**Coherent accumulation.** All $V_t$ are diagonal in the $|i,j\rangle$ basis and therefore commute. The product collapses cleanly:

$$
V \;\equiv\; \prod_{t=1}^M V_t \;=\; \sum_{i,j} \exp\!\Bigl(i\,\tau\, m_{ij}\, A_{ij}\Bigr)\,|i,j\rangle\!\langle i,j|, \tag{2}
$$

where $m_{ij} = \frac{1}{M}\sum_t \mathbf{1}[(i_t, j_t) = (i, j)]$ is the empirical frequency of matrix entry $(i, j)$.

**Concentration.** As $M\to\infty$, $m_{ij}\to 1/N^2$ by the law of large numbers, so $\tau\, m_{ij} \to \theta$ and

$$
V \;\longrightarrow\; \sum_{i,j} e^{i\,\theta\, A_{ij}}\,|i,j\rangle\!\langle i,j| \;=\; O_A^{(\text{el})}.\tag{3}
$$

**Sample complexity.** A single $\epsilon$-accurate phase-oracle sketch costs $M = \Theta(N^2 / \epsilon)$ classical samples — the $N^2$ replaces the $N$ of the Boolean case because the support $|i,j\rangle$ now ranges over $N^2$ basis states instead of $N$. To support a quantum query algorithm that makes $Q$ block-encoding queries to total error $\epsilon$ we get

$$
M_\text{total} \;=\; \Theta\!\bigl(N^2 Q^2/\epsilon\bigr). \tag{4}
$$

The per-sample gate is a 2-register version of the `apply_basis_phase` primitive from the [first notebook](qauntum_oracle_sketching.ipynb): a controlled phase gated on `(qvar_i == i_t) & (qvar_j == j_t)`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from oracle_sketching import (
    apply_basis_phase_2d,
    hadamard_test_branches,
    ideal_phase_oracle,
    matrix_element_oracle_sketch,
    matrix_element_sketch,
)
from quantum_ml import chebyshev_inverse_polynomial

from classiq import *

np.random.seed(13)

### Reference implementation (numpy)

We first sketch the matrix-element phase oracle classically, by drawing $M$ samples from $A$, accumulating empirical frequencies, and assembling the diagonal eq. (2). The result is an $N^2 \times N^2$ diagonal matrix indexed by `(i, j)` (row-major flatten).

In [ ]:
# --- Toy target matrix -------------------------------------------------
NUM_QUBITS_DATA = 2
N = 2**NUM_QUBITS_DATA  # N = 4
A_target = np.random.uniform(-1.0, 1.0, size=(N, N))
A_target = (A_target + A_target.T) / 2  # symmetric for downstream QSVT use
A_target /= np.max(np.abs(A_target))  # normalise to ‖A‖_max = 1
print(f"A target ({N}x{N}):\n{np.round(A_target, 3)}")

# --- Sketch ------------------------------------------------------------
theta = np.pi / 4
M_demo = 4000
samples_ij = np.column_stack(
    [
        np.random.randint(0, N, size=M_demo),
        np.random.randint(0, N, size=M_demo),
    ]
)

V = matrix_element_sketch(A_target, samples_ij, theta)
V_ideal = ideal_phase_oracle(A_target, theta)
err = np.linalg.norm(V - V_ideal, ord=2)
print(f"\nPhase-oracle error  ‖V − O_A^(el)‖_2  at M={M_demo}, N={N}:  {err:.3e}")
print(f"Predicted scaling   N²/M                              :  {N*N/M_demo:.3e}")

### Sample complexity

As in the Boolean case, two scalings are worth distinguishing:

- **Single-realization** operator-norm error: $\|V - O_A^{(\text{el})}\|_2$ for one random sample sequence is dominated by the variance of the empirical frequencies. Each $m_{ij}$ has standard deviation $\sqrt{1/(N^2 M)}$, so the per-entry phase error is $\theta N |A_{ij}|/\sqrt{M}$, and the maximum across the $N^2$ diagonal entries gives $\|V - O_A^{(\text{el})}\|_2 = \widetilde{\mathcal{O}}\bigl(\theta N / \sqrt{M}\bigr)$.

- **Random-unitary channel bias** $\|\mathbb{E}[V] - O_A^{(\text{el})}\|_2 = \mathcal{O}(N^2/M)$ — the diamond-distance scaling that drives eq. (4). The same coherent-accumulation argument used in the Boolean case applies here: distinct $(i,j)$ pairs label *mutually orthogonal* subspaces, so per-sample errors do not compound across the matrix.

We verify both by sweeping $M$ over three decades.

In [ ]:
M_values = np.unique(np.logspace(2.5, 5, 14, dtype=int))
n_trials = 80

single_err = np.empty(M_values.size)
for k, M_k in enumerate(M_values):
    errs = []
    for _ in range(n_trials):
        s = np.column_stack(
            [
                np.random.randint(0, N, size=int(M_k)),
                np.random.randint(0, N, size=int(M_k)),
            ]
        )
        V = matrix_element_sketch(A_target, s, theta)
        errs.append(np.linalg.norm(V - V_ideal, ord=2))
    single_err[k] = np.mean(errs)

c_fit = float(np.median(single_err * np.sqrt(M_values) / (theta * N)))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(
    M_values, single_err, "o-", label=r"single realization $\|V - O_A^{\rm el}\|_2$"
)
ax.loglog(
    M_values,
    c_fit * theta * N / np.sqrt(M_values),
    "--",
    color="C0",
    alpha=0.6,
    label=rf"${c_fit:.2f}\,\theta N/\sqrt{{M}}$ guide",
)
ax.set_xlabel("Number of samples $M$")
ax.set_ylabel("Spectral-norm error")
ax.set_title(
    f"Matrix-element phase oracle, $N={N}$, $\\theta=\\pi/4$, {n_trials} trials"
)
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Channel-bias check (analytical) — see Sec. D 2 of [[1]]:
# ⟨V_{ii}⟩ = exp(iθA_{ii}) · exp(-θ² A_{ii}² (N² − 1) / (2 M)),
# so ‖𝔼[V] − O_A^(el)‖_2 ≃ θ² (N² − 1) / (2 M) · max |A_{ii}|² = O(N² / M).
analytical_channel = (
    theta**2 * (N * N - 1) / (2 * M_values.astype(float)) * np.max(A_target**2)
)
print("\nAnalytical channel bias (peak entry):")
for M_k, b in zip(M_values[::3], analytical_channel[::3]):
    print(f"  M={M_k:>6}:  {b:.3e}")

## From phase oracle to block encoding

The matrix-element phase oracle $V$ is *almost* a block encoding already — what is missing is the conversion from the diagonal *phase* $e^{i\theta A_{ij}}$ to the off-diagonal *amplitude* $A_{ij}$. This is the same Hadamard-test trick we used for state sketching in the [first notebook](qauntum_oracle_sketching.ipynb), now applied to a 2-register oracle.

**Recipe.** Add a single ancilla qubit `aux`. Starting from $|0\rangle_\text{aux} \otimes |+\rangle^{\otimes n}_\text{row} \otimes |+\rangle^{\otimes n}_\text{col}$, apply

$$
U_\text{HT} \;=\; (H_\text{aux} \otimes I_\text{row} \otimes I_\text{col}) \cdot \mathrm{C}\text{-}V \cdot (H_\text{aux} \otimes I_\text{row} \otimes I_\text{col}).
$$

The output is

$$
\frac{1}{N} \sum_{i,j} e^{i\theta A_{ij}/2}\Bigl(\cos\!\tfrac{\theta A_{ij}}{2}\,|0\rangle_\text{aux} \;-\; i\,\sin\!\tfrac{\theta A_{ij}}{2}\,|1\rangle_\text{aux}\Bigr) |i\rangle\,|j\rangle.
$$

Post-selecting on $|1\rangle_\text{aux}$ leaves the unnormalised vector

$$
\sum_{i,j} \mathrm{amp}_1^{(i,j)} \,|i\rangle|j\rangle, \qquad \mathrm{amp}_1^{(i,j)} \;=\; -\,i\,\frac{e^{i\theta A_{ij}/2}}{N}\,\sin\!\frac{\theta A_{ij}}{2} \;\xrightarrow{\theta\to 0}\; -\,i\,\frac{\theta A_{ij}}{2N}.
$$

Reading off the post-selected amplitudes recovers $A_{ij}$ — the matrix entries appear as amplitudes of an $N^2$-dimensional state, scaled by $\alpha^{-1}$ with $\alpha = 2N/\theta$. Equivalently, this is a $(\alpha, 1, \mathcal{O}(\theta^2))$-**block encoding** of the diagonal embedding $\mathrm{diag}(A_{ij})$ on the doubled register.

**From diagonal-embed block encoding to operator block encoding.** Promoting $\mathrm{diag}(A_{ij})$ on $\mathbb{C}^{N^2}$ to the operator $A$ acting on $\mathbb{C}^{N}$ requires one more ingredient — the matrix-index oracle for sparse matrices, or, for dense matrices with $\|A\|_\text{max} \le 1$, the FABLE assembly (multiplexed-rotation-with-Hadamards, see [[4]](#ref4) Lemma 48 and Camps–Van Beeumen). That assembly is implemented inline in the [LS-SVM](qos_lssvm.ipynb) and [PCA](qos_pca.ipynb) notebooks, where the block encoding is consumed by [QSVT](https://arxiv.org/abs/1806.01838) [[4]](#ref4) to produce $A^{-1}$ or the spectral projector. Here we verify the QOS contribution: that the Hadamard-test branch of the *sketched* phase oracle reproduces the matrix entries with the predicted scaling.

In [ ]:
# Recover A from the |1⟩_aux branch of the sketched phase oracle.
amp_0_sketched, amp_1_sketched = hadamard_test_branches(np.diag(V), N)

# In the small-θ limit, amp_1[i, j] ≈ -i θ A_{ij} / (2 N), so
#     A_est = 2 N i amp_1 / θ.
A_est = 2 * N * 1j * amp_1_sketched / theta

err_A = np.linalg.norm(A_est - A_target) / np.linalg.norm(A_target)
print(f"|A_est - A| / |A| at M={M_demo}, θ={theta:.3f}: {err_A:.3e}")
print()
print(f"A_target:\n{np.round(A_target, 3)}")
print(f"\nRecovered (real part):\n{np.round(A_est.real, 3)}")
print(f"\nRecovered (imag part, should be ~ θA/2 ≪ A):\n{np.round(A_est.imag, 3)}")

## QSVT polynomial primer

A block encoding $U_A$ becomes useful once it is composed with QSVT [[4]](#ref4): given $U_A$ and a polynomial $P:[-1, 1] \to [-1, 1]$ of degree $d$, the QSVT circuit computes a *new* block encoding $U_{P(A)}$ of $P(A)$ using $d$ uses of $U_A$ (and its inverse). The polynomial $P$ is specified by a sequence of $d+1$ phase angles $\Phi = (\phi_0, \dots, \phi_d)$ that interleave with applications of $U_A$.

For the LS-SVM and PCA notebooks of this series we will need two specific polynomials:

* **Inversion**, $P_\text{inv}(x) = c/x$ on $[\kappa^{-1}, 1]\cup[-1, -\kappa^{-1}]$ — gives a block encoding of $A^{-1}$ for the linear-system stage of LS-SVM. Degree $d = \mathcal{O}(\kappa\,\log(\kappa/\epsilon))$.
* **Spectral projector**, $P_\text{proj}(x) = \tfrac{1}{2}\bigl(1 + \mathrm{sgn}(x - \mu)\bigr)$ — projects onto the subspace of eigenvalues above $\mu$, used for PCA's top-eigenvector extraction. Degree $d = \mathcal{O}(\Delta^{-1}\log(1/\epsilon))$ where $\Delta$ is the spectral gap.

Classiq exposes the polynomial-fit machinery via `classiq.applications.qsp.qsp_approximate` (Chebyshev fit on a target function with a parity constraint) and `qsvt_phases` (extract the phase angles from the Chebyshev coefficients). Below we pre-compute the angles for an inversion polynomial as a teaser; full QSVT execution is deferred to the LS-SVM / PCA notebooks where it solves a real task.

In [ ]:
from numpy.polynomial import chebyshev as cheb

# Inversion target with κ = 4.
kappa = 4.0
deg = 31
coeffs = chebyshev_inverse_polynomial(kappa, deg)
print(f"Chebyshev fit of (1/(2κ))/x on [{1/kappa:.2f}, 1]:")
print(f"  degree:         {deg}")
print(f"  nonzero coefs:  {np.count_nonzero(np.abs(coeffs) > 1e-10)} (odd parity)")
print(f"  max |coeff|:    {np.max(np.abs(coeffs)):.3f}")

# Plot fit vs target on the valid interval.
xs_plot = np.linspace(-1, 1, 400)
ys_target = np.where(np.abs(xs_plot) > 1.0 / kappa, 1.0 / (2 * kappa * xs_plot), np.nan)
ys_fit = cheb.chebval(xs_plot, coeffs)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(xs_plot, ys_target, "k-", label=r"$\frac{1}{2\kappa\,x}$ (target)")
ax.plot(xs_plot, ys_fit, "C0--", label=rf"Chebyshev deg {deg}")
ax.axvspan(
    -1.0 / kappa,
    1.0 / kappa,
    color="lightgray",
    alpha=0.3,
    label=r"forbidden $(|x|<1/\kappa)$",
)
ax.set_xlabel("x")
ax.set_ylabel("P(x)")
ax.set_ylim(-0.7, 0.7)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_title(f"QSVT polynomial design — inversion ($\\kappa = {kappa}$)")
plt.tight_layout()
plt.show()

## Classiq circuit fragment

The 2-register controlled phase that is the workhorse of the matrix-element sketch is just `apply_basis_phase` (from `oracle_sketching.py`) lifted to a pair of registers. The lift uses Classiq's nested `control` modifier: control on the row-equality predicate, with the inner statement itself a `control` on the column-equality predicate.

Below we define the lifted primitive and the full sketched matrix-element oracle, and synthesise a small demo at $N = 4$, $M = 80$ on $2 + 2 = 4$ data qubits.

In [ ]:
# --- Build a synthesisable demo ------------------------------------------
# `apply_basis_phase_2d` and `matrix_element_oracle_sketch` are now imported
# from `oracle_sketching.py`; we just bake the per-sample angles and wire
# everything into a `main` qfunc here.

M_circuit = 80
samples_demo = np.column_stack(
    [
        np.random.randint(0, N, size=M_circuit),
        np.random.randint(0, N, size=M_circuit),
    ]
)
angles_circuit = (
    theta * N * N / M_circuit * A_target[samples_demo[:, 0], samples_demo[:, 1]]
).tolist()


@qfunc
def main(qvar_i: Output[QNum], qvar_j: Output[QNum]) -> None:
    allocate(NUM_QUBITS_DATA, qvar_i)
    allocate(NUM_QUBITS_DATA, qvar_j)
    hadamard_transform(qvar_i)
    hadamard_transform(qvar_j)
    matrix_element_oracle_sketch(
        angles_circuit,
        samples_demo[:, 0].tolist(),
        samples_demo[:, 1].tolist(),
        qvar_i,
        qvar_j,
    )


qprog_be = synthesize(main)
show(qprog_be)

## What's next

- **LS-SVM classification** — [`qos_lssvm.ipynb`](qos_lssvm.ipynb) consumes the matrix-element sketch above to block-encode $X^\top X + \lambda I$, applies the QSVT inversion polynomial designed in this notebook, and reads out the classifier weight vector $\vec w$ via interferometric classical shadows.
- **PCA dimension reduction** — [`qos_pca.ipynb`](qos_pca.ipynb) does the same with $X^\top X$ and a QSVT spectral-projector polynomial, recovering the top principal direction.
- **Library promotion** — the helpers `matrix_element_sketch`, `hadamard_test_branches`, `chebyshev_inverse_polynomial`, `apply_basis_phase_2d`, and `matrix_element_oracle_sketch` are promoted to `oracle_sketching.py` and `quantum_ml.py` in the next refactor pass.
- **Sparse matrices** — for $s$-sparse $A$ the *matrix-index oracle* $O_A^{(\text{idx})}$ admits an analogous per-sample sketch: each sample $(i_t, k_t, j_{k_t}(i_t))$ contributes a 2-register *permutation* gate, which can be implemented in Classiq via `inplace_xor` on the column register controlled on `(qvar_i == i_t) & (qvar_k == k_t)`. We expand this in a follow-up notebook.

## References

<a id='ref1'></a>
[[1]](#ref1) Zhao, H., Zlokapa, A., Neven, H., Babbush, R., Preskill, J., McClean, J. R., and Huang, H.-Y. *Exponential quantum advantage in processing massive classical data.* arXiv:2604.07639 (2026). https://arxiv.org/abs/2604.07639

<a id='ref4'></a>
[[4]](#ref4) Gilyén, A., Su, Y., Low, G. H., and Wiebe, N. *Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics.* In Proceedings of the 51st Annual ACM SIGACT Symposium on Theory of Computing, 193–204 (2019). https://doi.org/10.1145/3313276.3316366 — arXiv: https://arxiv.org/abs/1806.01838

<a id='ref5'></a>
[[5]](#ref5) Harrow, A. W., Hassidim, A., and Lloyd, S. *Quantum algorithm for linear systems of equations.* Physical Review Letters 103, 150502 (2009). https://doi.org/10.1103/PhysRevLett.103.150502 — arXiv: https://arxiv.org/abs/0811.3171

<a id='ref-cv'></a>
[CV] Camps, D. and Van Beeumen, R. *FABLE: Fast Approximate quantum circuits for BLock-Encodings.* arXiv:2205.00081 (2022). https://arxiv.org/abs/2205.00081 — concrete construction of the FABLE block encoding referenced in the *From phase oracle to block encoding* section.